# Limpieza Profunda de Datos
## Dataset: Properatti
**Objetivo:** Corregir inconsistencias, valores erróneos y variables categóricas para dejar el dataset listo para Feature Engineering.

In [2]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset limpio del EDA
df = pd.read_csv('properatti_limpio.csv')

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 92445
Columnas: 19


,operation,property_type,place_name,place_with_parent_names,country_name,state_name,lat,lon,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,properati_url,description,title
0,sell,PH,Mataderos,|Argentina|Capital Federal|Mataderos|,Argentina,Capital Federal,-34.661824,-58.508839,62000.0,USD,1093959.0,62000.0,55.0,40.0,1127.272727,1550.000000,http://www.properati.com.ar/15bo8_venta_ph_mat...,"2 AMBIENTES TIPO CASA PLANTA BAJA POR PASILLO,...",2 AMB TIPO CASA SIN EXPENSAS EN PB
1,sell,apartment,La Plata,|Argentina|Bs.As. G.B.A. Zona Sur|La Plata|,Argentina,Bs.As. G.B.A. Zona Sur,-34.903883,-57.964330,150000.0,USD,2646675.0,150000.0,NaN,NaN,NaN,NaN,http://www.properati.com.ar/15bob_venta_depart...,Venta de departamento en décimo piso al frente...,VENTA Depto 2 dorm. a estrenar 7 e/ 36 y 37 ...
2,sell,apartment,Mataderos,|Argentina|Capital Federal|Mataderos|,Argentina,Capital Federal,-34.652262,-58.522982,72000.0,USD,1270404.0,72000.0,55.0,55.0,1309.090909,1309.090909,http://www.properati.com.ar/15bod_venta_depart...,2 AMBIENTES 3ER PISO LATERAL LIVING COMEDOR AM...,2 AMB 3ER PISO CON ASCENSOR APTO CREDITO
3,sell,PH,Liniers,|Argentina|Capital Federal|Liniers|,Argentina,Capital Federal,-34.647797,-58.516424,95000.0,USD,1676227.5,95000.0,NaN,NaN,NaN,NaN,http://www.properati.com.ar/15boh_venta_ph_lin...,PH 3 ambientes con patio. Hay 3 deptos en lote...,PH 3 amb. cfte. reciclado
4,sell,apartment,Centro,|Argentina|Buenos Aires Costa Atlántica|Mar de...,Argentina,Buenos Aires Costa Atlántica,-38.002626,-57.549447,64000.0,USD,1129248.0,64000.0,35.0,35.0,1828.571429,1828.571429,http://www.properati.com.ar/15bok_venta_depart...,DEPARTAMENTO CON FANTÁSTICA ILUMINACIÓN NATURA...,DEPTO 2 AMB AL CONTRAFRENTE ZONA CENTRO/PLAZA ...


In [3]:
# Revisar variables categóricas
categoricas = df.select_dtypes(include='object').columns.tolist()
print("Variables categóricas:", categoricas)
print()

for col in categoricas:
    print(f"--- {col} ---")
    print(df[col].value_counts().head(10))
    print()

Variables categóricas: ['operation', 'property_type', 'place_name', 'place_with_parent_names', 'country_name', 'state_name', 'currency', 'properati_url', 'description', 'title']

--- operation ---
operation
sell    92445
Name: count, dtype: int64

--- property_type ---
property_type
apartment    56477
house        28233
PH            5166
store         2569
Name: count, dtype: int64

--- place_name ---
place_name
Córdoba          6413
Mar del Plata    5914
Rosario          4594
Tigre            2711
Nordelta         2254
Palermo          2128
Belgrano         2114
Caballito        2013
Pilar            1775
La Plata         1651
Name: count, dtype: int64

--- place_with_parent_names ---
place_with_parent_names
|Argentina|Buenos Aires Costa Atlántica|Mar del Plata|    5914
|Argentina|Santa Fe|Rosario|                              4594
|Argentina|Córdoba|Córdoba|                               4364
|Argentina|Bs.As. G.B.A. Zona Norte|Tigre|Nordelta|       2254
|Argentina|Capital Federal|P

In [4]:
# Eliminar columnas que no aportan al modelo
columnas_eliminar = [
    'operation',
    'country_name', 
    'place_with_parent_names',
    'properati_url',
    'description',
    'title'
]

df = df.drop(columns=columnas_eliminar)

# Quedarnos solo con propiedades en USD
df = df[df['currency'] == 'USD']

# Eliminar columna currency (ya no necesaria)
df = df.drop(columns=['currency'])

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas restantes: {df.columns.tolist()}")

Filas: 79280
Columnas: 12

Columnas restantes: ['property_type', 'place_name', 'state_name', 'lat', 'lon', 'price', 'price_aprox_local_currency', 'price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2', 'price_usd_per_m2', 'price_per_m2']


In [5]:
# Verificar nulos restantes
nulos = pd.DataFrame({
    'Valores nulos': df.isnull().sum(),
    'Porcentaje (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos['Valores nulos'] > 0].sort_values('Porcentaje (%)', ascending=False)
print(nulos)

                       Valores nulos  Porcentaje (%)
lat                            31141           39.28
lon                            31141           39.28
price_usd_per_m2               25336           31.96
surface_total_in_m2            25151           31.72
price_per_m2                    9749           12.30
surface_covered_in_m2           9747           12.29
place_name                         8            0.01


### Decisiones de limpieza final
- Se eliminan lat, lon: alta proporción de nulos y baja correlación con precio
- Se eliminan price_usd_per_m2 y price_per_m2: fuga de datos
- Se imputa surface_total_in_m2 y surface_covered_in_m2 con la mediana
- Se eliminan 8 filas nulas de place_name

In [6]:
# Eliminar columnas con fuga de datos o alta proporción de nulos
df = df.drop(columns=['lat', 'lon', 'price_usd_per_m2', 'price_per_m2', 'price', 'price_aprox_local_currency'])

# Imputar surface con la mediana
df['surface_total_in_m2'] = df['surface_total_in_m2'].fillna(df['surface_total_in_m2'].median())
df['surface_covered_in_m2'] = df['surface_covered_in_m2'].fillna(df['surface_covered_in_m2'].median())

# Eliminar 8 filas nulas de place_name
df = df.dropna(subset=['place_name'])

# Verificar que no queden nulos
print("Nulos restantes:")
print(df.isnull().sum())
print(f"\nFilas finales: {df.shape[0]}")
print(f"Columnas finales: {df.shape[1]}")
print(f"\nColumnas: {df.columns.tolist()}")

Nulos restantes:
property_type            0
place_name               0
state_name               0
price_aprox_usd          0
surface_total_in_m2      0
surface_covered_in_m2    0
dtype: int64

Filas finales: 79272
Columnas finales: 6

Columnas: ['property_type', 'place_name', 'state_name', 'price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2']


In [7]:
# Guardar dataset preparado
df.to_csv('properatti_preparado.csv', index=False)
print("Dataset guardado correctamente")

Dataset guardado correctamente
